# RNN基础与文本序列处理

本notebook介绍序列数据和循环神经网络:
- **序列模型**: 为什么需要处理序列?
- **文本预处理**: 分词、词表、数值化
- **RNN原理**: 隐状态与循环结构
- **从零实现RNN**: 字符级语言模型

理解RNN是学习LSTM/GRU/Transformer的基础!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import time
import re
import collections
import random

---

## 第一部分: 为什么需要序列模型?

### 1.1 序列数据无处不在

**什么是序列数据?**
- **顺序很重要**的数据
- 打乱顺序会改变意义

**例子**:

| 领域 | 序列数据 | 为什么顺序重要? |
|------|----------|----------------|
| **文本** | "狗咬人" vs "人咬狗" | 完全不同的新闻! |
| **音乐** | do-re-mi vs mi-re-do | 不同的旋律 |
| **视频** | 电影帧序列 | 打乱就看不懂 |
| **时间序列** | 股价、气温 | 历史影响未来 |
| **语音** | 声音波形 | 顺序决定内容 |

### 1.2 序列预测问题

**核心任务**: 根据历史预测未来

$$P(x_t \mid x_{t-1}, x_{t-2}, ..., x_1)$$

**例子**:
- **语言模型**: "今天天气真" → 预测下一个字
- **股价预测**: 根据过去30天预测明天
- **机器翻译**: 根据源语言预测目标语言

### 1.3 为什么CNN/MLP不够?

**问题1: 输入长度可变**
```python
sentence1 = "我爱你"        # 3个字
sentence2 = "今天天气真好啊" # 7个字
# CNN/MLP需要固定输入长度!
```

**问题2: 参数爆炸**
```python
# 如果考虑前100个词
vocab_size = 10000
params = vocab_size ** 100  # 天文数字!
```

**问题3: 无法共享参数**
- "猫在树上" 和 "狗在树上"
- "在树上"应该共享知识
- 但MLP把每个位置当独立参数

**RNN的解决方案**:
1. ✅ 处理任意长度序列
2. ✅ 参数在时间步间共享
3. ✅ 维护"记忆"(隐状态)

---

## 第二部分: 文本预处理

### 2.1 文本到数字的流程

```
原始文本 → 分词 → 构建词表 → 数值化 → 张量
"我爱你"  [我,爱,你]  {我:0,爱:1,你:2}  [0,1,2]  tensor
```

### 2.2 实现文本预处理

In [ ]:
def read_time_machine():
    """加载时间机器数据集(示例文本)"""
    # 这里用简单的中文文本示例
    text = """深度学习是机器学习的一个分支。
机器学习研究如何让计算机从数据中学习。
深度学习使用神经网络来学习数据的表示。
循环神经网络可以处理序列数据。
序列数据包括文本、语音和时间序列。"""
    return text


def tokenize(text, token='char'):
    """分词: 字符级或词级"""
    if token == 'char':
        # 字符级: 每个字符是一个token
        return list(text.replace('\n', ' ').replace('\t', ' '))
    elif token == 'word':
        # 词级: 简单按空格分词(中文需要jieba等工具)
        return text.split()


# 测试
text = read_time_machine()
print("原始文本:")
print(text[:100])

tokens_char = tokenize(text, 'char')
print(f"\n字符级分词 (前30个): {tokens_char[:30]}")
print(f"总共{len(tokens_char)}个字符")

In [ ]:
class Vocab:
    """词表类"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        
        # 统计词频
        counter = collections.Counter(tokens)
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        
        # 构建词到索引的映射
        # 保留特殊token: <unk>(未知词), <pad>(填充), <bos>(开始), <eos>(结束)
        self.unk = 0  # 未知词索引
        self.token_to_idx = {'<unk>': 0}
        self.idx_to_token = ['<unk>']
        
        # 添加保留token
        for token in reserved_tokens:
            self.idx_to_token.append(token)
            self.token_to_idx[token] = len(self.idx_to_token) - 1
        
        # 添加高频词
        for token, freq in self.token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1
    
    def __len__(self):
        return len(self.idx_to_token)
    
    def __getitem__(self, tokens):
        """token → index"""
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
    
    def to_tokens(self, indices):
        """index → token"""
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]


# 构建词表
vocab = Vocab(tokens_char)
print(f"\n词表大小: {len(vocab)}")
print(f"前20个token: {vocab.idx_to_token[:20]}")

# 测试转换
test_text = "深度学习"
indices = vocab[list(test_text)]
print(f"\n'{test_text}' → {indices}")
print(f"{indices} → {vocab.to_tokens(indices)}")

### 2.3 数据加载器

In [ ]:
def seq_data_iter_random(corpus, batch_size, num_steps):
    """随机采样迭代器"""
    # 随机偏移开始位置
    corpus = corpus[random.randint(0, num_steps - 1):]
    # 可用的序列数量
    num_subseqs = (len(corpus) - 1) // num_steps
    # 起始索引
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)
    
    def data(pos):
        return corpus[pos: pos + num_steps]
    
    num_batches = num_subseqs // batch_size
    for i in range(0, batch_size * num_batches, batch_size):
        # 每个batch的起始索引
        initial_indices_per_batch = initial_indices[i: i + batch_size]
        X = [data(j) for j in initial_indices_per_batch]
        Y = [data(j + 1) for j in initial_indices_per_batch]
        yield torch.tensor(X), torch.tensor(Y)


def seq_data_iter_sequential(corpus, batch_size, num_steps):
    """顺序分区迭代器"""
    # 随机偏移
    offset = random.randint(0, num_steps)
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    Xs = torch.tensor(corpus[offset: offset + num_tokens])
    Ys = torch.tensor(corpus[offset + 1: offset + num_tokens + 1])
    Xs, Ys = Xs.reshape(batch_size, -1), Ys.reshape(batch_size, -1)
    num_batches = Xs.shape[1] // num_steps
    for i in range(0, num_steps * num_batches, num_steps):
        X = Xs[:, i: i + num_steps]
        Y = Ys[:, i: i + num_steps]
        yield X, Y


class SeqDataLoader:
    """加载序列数据的迭代器"""
    def __init__(self, batch_size, num_steps, use_random_iter, max_tokens):
        if use_random_iter:
            self.data_iter_fn = seq_data_iter_random
        else:
            self.data_iter_fn = seq_data_iter_sequential
        self.corpus, self.vocab = self.load_corpus(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps
    
    def load_corpus(self, max_tokens):
        """加载语料库"""
        text = read_time_machine()
        tokens = tokenize(text, 'char')
        vocab = Vocab(tokens)
        corpus = [vocab[token] for token in tokens]
        if max_tokens > 0:
            corpus = corpus[:max_tokens]
        return corpus, vocab
    
    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)


# 测试数据加载器
data_loader = SeqDataLoader(batch_size=2, num_steps=5, use_random_iter=False, max_tokens=1000)
print(f"\n词表大小: {len(data_loader.vocab)}")

for X, Y in data_loader:
    print(f"X shape: {X.shape}, Y shape: {Y.shape}")
    print(f"X[0]: {X[0]}")
    print(f"Y[0]: {Y[0]}")
    print(f"文本: {''.join(data_loader.vocab.to_tokens(X[0].tolist()))}")
    break

---

## 第三部分: RNN原理

### 3.1 从MLP到RNN

**MLP (无记忆)**:
$$\mathbf{H} = \phi(\mathbf{X} \mathbf{W}_{xh} + \mathbf{b}_h)$$

**RNN (有记忆)**:
$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h)$$

**关键变化**:
- 增加了 $\mathbf{H}_{t-1} \mathbf{W}_{hh}$ 项
- $\mathbf{H}_{t-1}$: 上一时刻的隐状态("记忆")
- $\mathbf{W}_{hh}$: 隐状态到隐状态的权重

### 3.2 RNN展开图

```
时间步:    t-1           t            t+1
           ↓             ↓             ↓
输入:     x_{t-1}       x_t          x_{t+1}
           ↓             ↓             ↓
隐状态:  H_{t-1} ----→ H_t  ----→  H_{t+1}
           ↓             ↓             ↓
输出:     y_{t-1}       y_t          y_{t+1}
```

**循环结构**:
- 同一个网络在每个时间步重复使用
- 参数 $\mathbf{W}_{xh}, \mathbf{W}_{hh}$ 在所有时间步共享
- 隐状态 $\mathbf{H}_t$ 携带历史信息

### 3.3 RNN的输出

$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q$$

**语言模型**:
$$\hat{\mathbf{Y}}_t = \text{softmax}(\mathbf{O}_t)$$

预测下一个词的概率分布。

### 3.4 RNN的参数

| 参数 | 形状 | 作用 |
|------|------|------|
| $\mathbf{W}_{xh}$ | $(d, h)$ | 输入到隐状态 |
| $\mathbf{W}_{hh}$ | $(h, h)$ | 隐状态到隐状态 |
| $\mathbf{W}_{hq}$ | $(h, q)$ | 隐状态到输出 |
| $\mathbf{b}_h$ | $(h,)$ | 隐状态偏置 |
| $\mathbf{b}_q$ | $(q,)$ | 输出偏置 |

其中:
- $d$: 输入维度(词表大小)
- $h$: 隐藏层大小
- $q$: 输出维度(词表大小)

---

## 第四部分: 从零实现RNN

### 4.1 初始化参数

In [ ]:
def get_params(vocab_size, num_hiddens, device):
    """初始化RNN参数"""
    num_inputs = num_outputs = vocab_size
    
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    
    # 隐藏层参数
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    
    # 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    
    # 附加梯度
    params = [W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params


# 测试
vocab_size = len(data_loader.vocab)
num_hiddens = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
params = get_params(vocab_size, num_hiddens, device)
print(f"参数数量: {sum(p.numel() for p in params):,}")
print(f"W_xh shape: {params[0].shape}")
print(f"W_hh shape: {params[1].shape}")
print(f"W_hq shape: {params[3].shape}")

### 4.2 RNN前向传播

In [ ]:
def init_rnn_state(batch_size, num_hiddens, device):
    """初始化隐状态"""
    return (torch.zeros((batch_size, num_hiddens), device=device),)


def rnn(inputs, state, params):
    """RNN前向传播
    
    inputs: (num_steps, batch_size, vocab_size) - one-hot编码
    state: (batch_size, num_hiddens)
    """
    W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state  # 解包
    outputs = []
    
    # 遍历每个时间步
    for X in inputs:
        # H_t = tanh(X_t @ W_xh + H_{t-1} @ W_hh + b_h)
        H = torch.tanh(torch.mm(X, W_xh) + torch.mm(H, W_hh) + b_h)
        # Y_t = H_t @ W_hq + b_q
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    
    # 拼接所有输出: (num_steps * batch_size, vocab_size)
    return torch.cat(outputs, dim=0), (H,)


# 测试前向传播
X = torch.arange(10).reshape((2, 5))  # batch_size=2, num_steps=5
state = init_rnn_state(X.shape[0], num_hiddens, device)

# one-hot编码
X_one_hot = F.one_hot(X.T.long(), vocab_size).type(torch.float32).to(device)
print(f"输入形状: {X_one_hot.shape}")  # (num_steps, batch_size, vocab_size)

Y, new_state = rnn(X_one_hot, state, params)
print(f"输出形状: {Y.shape}")  # (num_steps * batch_size, vocab_size)
print(f"新隐状态形状: {new_state[0].shape}")  # (batch_size, num_hiddens)

### 4.3 预测函数

In [ ]:
def predict(prefix, num_preds, net, vocab, device):
    """生成文本
    
    prefix: 起始字符串
    num_preds: 生成字符数
    """
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    
    # 用prefix warm-up模型
    for i in range(len(prefix) - 1):
        X = torch.tensor([outputs[-1]], device=device).reshape((1, 1))
        X = F.one_hot(X.T.long(), len(vocab)).type(torch.float32)
        Y, state = net(X, state)
        outputs.append(vocab[prefix[i + 1]])
    
    # 预测num_preds个字符
    for _ in range(num_preds):
        X = torch.tensor([outputs[-1]], device=device).reshape((1, 1))
        X = F.one_hot(X.T.long(), len(vocab)).type(torch.float32)
        Y, state = net(X, state)
        # 贪心选择概率最大的
        outputs.append(int(Y.argmax(dim=1)))
    
    return ''.join([vocab.idx_to_token[i] for i in outputs])


# 包装成类
class RNNModelScratch:
    """从零实现的RNN模型"""
    def __init__(self, vocab_size, num_hiddens, device, get_params_fn, init_state_fn, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params_fn(vocab_size, num_hiddens, device)
        self.init_state_fn, self.forward_fn = init_state_fn, forward_fn
    
    def __call__(self, X, state):
        X = F.one_hot(X.T.long(), self.vocab_size).type(torch.float32).to(X.device)
        return self.forward_fn(X, state, self.params)
    
    def begin_state(self, batch_size, device):
        return self.init_state_fn(batch_size, self.num_hiddens, device)


# 创建模型
net = RNNModelScratch(vocab_size, num_hiddens, device, get_params, init_rnn_state, rnn)

# 测试预测(未训练)
print("\n未训练的预测(随机):")
print(predict('深度学习', 20, net, data_loader.vocab, device))

### 4.4 训练RNN

**梯度裁剪**: 防止梯度爆炸

$$\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right) \mathbf{g}$$

In [ ]:
def grad_clipping(net, theta):
    """梯度裁剪"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm


def train_epoch(net, train_iter, loss, updater, device, use_random_iter):
    """训练一个epoch"""
    state = None
    metric = [0.0, 0.0]  # 总损失, token数
    
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            # 分离隐状态(不计算梯度)
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state.detach_()
            else:
                for s in state:
                    s.detach_()
        
        y = Y.T.reshape(-1).to(device)
        X = X.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        
        metric[0] += l * y.numel()
        metric[1] += y.numel()
    
    return math.exp(metric[0] / metric[1])


def train(net, train_iter, vocab, lr, num_epochs, device, use_random_iter=False):
    """训练模型"""
    import math
    loss = nn.CrossEntropyLoss()
    
    # SGD优化器
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    
    # 训练
    for epoch in range(num_epochs):
        ppl = train_epoch(net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch + 1}, perplexity {ppl:.1f}')
            print(predict('深度学习', 50, net, vocab, device))
    
    print(f'\nperplexity {ppl:.1f}')
    print(predict('深度学习', 50, net, vocab, device))
    print(predict('循环神经', 50, net, vocab, device))


def sgd(params, lr, batch_size):
    """小批量SGD"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()


# 训练(小规模示例)
import math
print("\n开始训练...")
num_epochs, lr = 50, 1
train(net, data_loader, data_loader.vocab, lr, num_epochs, device)

---

## 小结

### RNN核心概念

1. **序列数据**: 顺序很重要的数据(文本、时间序列等)

2. **隐状态**: RNN的"记忆"
   $$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h)$$

3. **参数共享**: 所有时间步共享参数

4. **字符级语言模型**: 预测下一个字符

### 文本预处理流程

```python
文本 → 分词 → 构建词表 → 数值化 → 批处理
```

### 训练技巧

1. **梯度裁剪**: 防止梯度爆炸
2. **困惑度(Perplexity)**: 评估语言模型
   $$\text{PPL} = \exp\left(-\frac{1}{n}\sum_{t=1}^n \log P(x_t \mid x_{t-1}, ...)\right)$$
3. **隐状态分离**: 截断反向传播

### RNN的问题

- ❌ **梯度消失/爆炸**: 难以学习长期依赖
- ❌ **顺序计算**: 无法并行
- ❌ **记忆有限**: 长序列信息丢失

**解决方案**: LSTM、GRU (下一个notebook)

## 练习

1. 调整隐藏层大小,观察对困惑度的影响
2. 比较字符级和词级语言模型
3. 实现温度采样而不是贪心采样
4. 在更大的文本数据集上训练